In [46]:
# First generate all possibilities

import itertools as itr

C3_1_prim = [(1,0,0,0),(0,1,0,0),(0,0,1,0),(0,0,0,1)]
C3_2_prim = [(2,0,0,0),(0,2,0,0),(0,0,2,0),(0,0,0,2),\
             (1,1,0,0),(1,0,1,0),(1,0,0,1),(0,1,1,0),(0,1,0,1),(0,0,1,1)]

primaries = []
count_par = 0
count_dup = 0
for poss_prim in itr.product(C3_1_prim,C3_1_prim,C3_2_prim):
    
     # Check if selection rules are satisfied
    par = sum( j*poss_prim[i][j] for i,j in itr.product(range(3),range(1,4)) )
    if par%2:
        count_par+=1
        continue

    # Now check to make sure this primary isn't already in the set of primaries with different labels
    Aposs_prim = tuple(tuple(poss_prim[i][j] for j in range(3,-1,-1)) for i in range(3))
    if Aposs_prim in primaries:
        count_dup+=1
        continue

    # If the primary has passed all checks, add it to the list
    primaries.append(poss_prim)

In [54]:
# Now we wish to calculate the S-matrix

# To do this we need to first calculate the S-matrix of sp(6)_k
# To this end we recall the formula (K is some constant of proportionality):
# S_{\lambda,\mu} = K sum_{w\in W} sgn(w) exp(-\frac{2\pi i}{k+g^{\vee}} <w(\lambda+rho),\mu+\rho)>
# Note that, on the RHS, none of the objects are 'affine'

from sage.groups.matrix_gps.finitely_generated import MatrixGroup

# The rank of the algebra
rank = 3

# The dual Coxeter number
gvee = 4

# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# The Weyl vector
rho = vector([1,1,1])

# The simple roots, coroots, and weights are as follows
simp_roots = [vector([2,-1,0]),vector([-1,2,-1]),vector([0,-2,2])]
simp_lengths = [(root*quadF*root)[0] for root in simp_roots]
simp_coroots = [2*simp_roots[i]/simp_lengths[i] for i in range(rank)]
simp_weights = [vector([1,0,0]),vector([0,1,0]),vector([0,0,1])]

# First let's find the Weyl group (not the affine weyl group)
# There are three simple Weyl reflections

# Calculate the matrix form of the simple Weyl reflections
simp_weyl = []
for i in range(rank):
    root = simp_roots[i]
    coroot = simp_coroots[i]
    # Find the columns of the matrix
    columns = [w - (coroot*quadF*w)[0]*root for w in simp_weights] 
    simp_weyl.append(matrix(columns).T)

# Use these to generate the full Weyl group
weyl_group = MatrixGroup(simp_weyl)
weyl_ref = [matrix(w) for w in list(weyl_group)]

# Now we are prepared to actually calculate the S-matrices of sp(6)_level
kac_C3_1 = [vector(l[1:]) for l in C3_1_prim]
kac_C3_2 = [vector(l[1:]) for l in C3_2_prim]
def S_matrix_sp6(level):
    K = I*8**(-1/2)*(level+gvee)**(-rank/2)
    if level==1:
        kac = kac_C3_1 
    elif level==2:
        kac = kac_C3_2
    S = matrix(SR,len(kac))
    for i,j in itr.product(range(len(kac)),repeat=2):
        S[i,j] = ( K*sum(w.det()*exp(-2*pi*I/(level+gvee)*((w*(kac[i]+rho))*quadF*(kac[j]+rho))[0]) for w in weyl_ref) ).simplify(algorithm='giac')
    
    return S

S1 = S_matrix_sp6(1)
S2 = S_matrix_sp6(2)

# Now we combine these into the S-matrix for the full theory
S_dsp6 = matrix(SR,len(primaries))
for i,j in itr.product(range(len(primaries)),repeat=2):
    pri = primaries[i]
    prj = primaries[j]
    S_dsp6[i,j] = ( 2*S1[C3_1_prim.index(pri[0]),C3_1_prim.index(prj[0])] * S1[C3_1_prim.index(pri[1]),C3_1_prim.index(prj[1])] * S2[C3_2_prim.index(pri[2]),C3_2_prim.index(prj[2])] ).simplify(algorithm='giac')

In [32]:
# Now let's calculate the characters

import itertools as itr

# Affine root system
LC31 = RootSystem(['C', 3, 1]).weight_lattice(extended=True)
# List of fundamental weights
LaC31 = LC31.fundamental_weights()
# The null root delta
deltC31 = LC31.null_root()
# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# Get the affine Weyl group and isolate the classical generators
affine_weyl = LC31.weyl_group()
classical_nodes = [i for i in LC31.index_set() if i != 0]
generators = [affine_weyl.simple_reflection(i) for i in classical_nodes]

# Get the classical Weyl group
classical_weyl = list( RootSystem(['C',3]).weight_lattice().weyl_group() ) # Get the group
classical_weyl = [w.matrix() for w in classical_weyl] # We want a matrix representation

def are_equivalent(w1, w2):
    '''Function to determine if two weights are related via shift by a coroot'''
    c1 = w1.monomial_coefficients()
    c2 = w2.monomial_coefficients()
    return all((c1.get(i, 0) - c2.get(i, 0)) % (2 * 1) == 0 for i in classical_nodes)

def max_mod_coroots(int_rep):
    '''Function that determines the maximal weights in the integrable representation mod the coroots'''
    # Every maximal weight is related to a dominant weight via a classical weyl transform
    dom_wghts = int_rep.dominant_maximal_weights()

    # We will now generate a full set of representations by applying classical weyl transforms
    full_maximal_representatives = list( dom_wghts )
    queue = list( dom_wghts )
    while queue:
        current = queue.pop(0)
        for g in generators:
            nxt = g.action(current)
            # Only keep nxt if it represents a brand new class modulo the coroots
            if not any(are_equivalent(nxt, r) for r in full_maximal_representatives):
                full_maximal_representatives.append(nxt)
                queue.append(nxt)
    return full_maximal_representatives


def C3_dcoset_char(lam, mu, nu, max_ord=10):
    '''
    Function to compute the coset character of C3_1xC3_1/C3_2.
    lam,mu,nu should all be triples of nonnegative integers.
    The entries of lam,mu should sum to no more than one, and the entries of nu to no more than two.
    Returns the coset character chi_{lam,mu;nu} of the diagonal coset.
    This is a power series in q with terms up to O(q^{max_ord})
    '''
    if sum(lam) > 1 or sum(mu) > 1 or sum(nu) > 2:
        raise ValueError('Must input affine weights with the correct levels.')

    # Define the rings that the chracters will live in
    P = PuiseuxSeriesRing(ZZ, 'q', sparse = True, default_prec = max_ord+1)
    q = P.gen()

    # Check selection rule before continuing
    if (lam[0]+lam[2]+mu[0]+mu[2]-nu[0]-nu[2])%2 != 0:
        return P(0)

    # The weight lambda in Sagemaths internal weight representation
    Lam = sum(lam[i]*LaC31[i+1] for i in range(3)) + (1-sum(lam))*LaC31[0]
    # This code computes all the string functions necessary for the character
    int_rep = IntegrableRepresentation(Lam)
    max_wghts = max_mod_coroots(int_rep)
   
    # Compute the strings and modular anomalies
    list_strings = {wght:int_rep.string(wght,depth=max_ord+1) for wght in max_wghts}
    mod_ch = {wght:int_rep.modular_characteristic(wght) for wght in max_wghts}
   
    # String functions in this ring
    strings = {wght: P(list_strings[wght])*q**(mod_ch[wght]) for wght in max_wghts}

    # Convert lam,mu,nu to vectors if they aren't already
    lam = vector(lam)
    mu = vector(mu)
    nu = vector(nu)

    # Define the Weyl vector
    rho = vector([1,1,1])

    char = P(0)
    char = char.add_bigoh(max_ord)
    for Gam in max_wghts:
        this_char = P(0)
        # This is a tuple containing the Dynkin labels
        gam = vector(Gam[i] for i in range(1,4))
        # Loop over the weyl group
        for wr in classical_weyl:
            # The sign of the element is given as its determinant in this representation
            sgn_wr = wr.det()
            for alpha in itr.product(range(-ceil(max_ord/12)*12,(ceil(max_ord/12)+1)*12,12),repeat=3):
                check_vec = wr*(rho+nu) - rho-mu-gam
                if not any(check_vec[i]%2 for i in range(3)):
                    alpha = vector(alpha)
                    evec = wr*(rho+nu) + alpha - (6/5)*(mu+rho)
                    expo = (5/6)*( evec*quadF*evec )[0]/2
                    if expo >=0 and expo <= max_ord:
                        this_char += sgn_wr*q**expo
        # Add this to the result
        char += this_char*strings[Gam] 
    
    # Return the full character
    return char

In [56]:
# Create a list with all the characters

all_dC3_char = list()
for lam,mu,nu in primaries:
    all_dC3_char.append( C3_dcoset_char(lam[1:], mu[1:], nu[1:], max_ord=15) )
    
C3_dcoset_char((0,0,0),(0,0,0),(0,0,0))


q^(-7/120) + q^(233/120) + q^(353/120) + 3*q^(473/120) + 3*q^(593/120) + 7*q^(713/120) + 8*q^(833/120) + 15*q^(953/120) + 19*q^(1073/120) + 30*q^(1193/120) + O(q^10)

In [52]:
c_wghts = list()
for char in all_dC3_char:
    c_wghts.append( char.valuation() + 7/5/24 )

T_dsp6 = diagonal_matrix( [ exp(2*pi*I*(w-7/5/24)) for w in c_wghts ] )


In [57]:
# Write these for Panos!
with open('m-dcoset-sp6-S.m','w') as f:
    f.write(mathematica(S_dsp6)._repr_())
with open('m-dcoset-sp6-T.m','w') as f:
    f.write(mathematica(T_dsp6)._repr_())
with open('m-dcoset-sp6-conformal-weights.m','w') as f:
    f.write(mathematica(c_wghts)._repr_())

In [6]:
# Now we calculate the T-matrix and, by extension, the conformal weights
# First we calculate the modular anomalies and T-matrix of sp(6)_1 and sp(6)_2
moda1 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(1+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_1 ]
moda2 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(2+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_2 ]
T1 = diagonal_matrix( [ exp( 2*pi*I*m ) for m in moda1 ] )
T2 = diagonal_matrix( [ exp( 2*pi*I*m ) for m in moda2 ] )

# Now calculate the T matrix of the coset
T_dsp6 = matrix(SR,len(primaries))
for i in range(len(primaries)):
    pri = primaries[i]
    T_dsp6[i,i] = ( T1[C3_1_prim.index(pri[0]),C3_1_prim.index(pri[0])] * T1[C3_1_prim.index(pri[1]),C3_1_prim.index(pri[1])] \
                / T2[C3_2_prim.index(pri[2]),C3_2_prim.index(pri[2])] )#.simplify(algorithm='giac')

In [6]:
print( (T1*T1.H-identity_matrix(4)).n().norm() )
print( ((S1.n()*T1.n())**3 - S1.n()**2).n().norm() )
print( S1.is_symmetric() )
print( (S1*S1.H-identity_matrix(4)).n().norm() )

0.0
1.0819089225884229e-15
True
5.551115123125783e-17


In [7]:
print( (T2*T2.H-identity_matrix(10)).n().norm() )
print( ((S2.n()*T2.H.n())**3 - S2.n()**2).n().norm() )
print( S2.is_symmetric() )
print( (S2*S2.H-identity_matrix(10)).n().norm() )

0.0
5.470990389089858e-16
True
1.856963405217746e-16


In [55]:
print( (T_dsp6*T_dsp6.H-identity_matrix(40)).n().norm() )
print( ((S_dsp6.n()*T_dsp6.n())**3 - S_dsp6.n()**2).n().norm() )
print( S_dsp6.is_symmetric() )
print( (S_dsp6*S_dsp6.H-identity_matrix(40)).n().norm() )

0.0
1.3336624236470513e-15
True
3.618476623959175e-16
